# Phase 1 · Baseline Evaluation

Goal: produce the first two points of the Pareto frontier — the **uncompressed** teacher (Llama-3.1-8B) and student (Llama-3.2-1B).

Each run records `{perplexity, tokens_per_sec, ttft_ms, peak_vram_gb}` into `results/runs/<ts>-<tag>/metrics.json`.

**Hardware assumption**: L4 (23 GB).
- Student fp16 → ~2 GB, easy.
- Teacher fp16 → ~16 GB, fits on L4. If OOM, fall back to int8.

⚠ Run `00_setup.ipynb` first in this session.

In [ ]:
# Make the repo importable. Adjust if your clone lives elsewhere.
import sys, os
REPO = '/content/llama'
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')

## 1 · Student baseline (Llama-3.2-1B, fp16)

In [ ]:
import torch, gc
from src.data.wikitext import load_wikitext_text
from src.eval.perplexity import compute_perplexity
from src.eval.latency import measure_latency
from src.eval.memory import peak_vram_gb, reset_peak
from src.models.loader import load_model, load_tokenizer
from src.utils import RunContext, load_yaml, new_run_dir, save_metrics

mcfg = load_yaml('configs/models.yaml')
ecfg = load_yaml('configs/eval.yaml')

tok = load_tokenizer(mcfg['tokenizer']['name'])
reset_peak()
student = load_model(mcfg['student']['name'], quant='none', dtype=mcfg['student']['dtype'])
print('loaded student')

In [ ]:
text = load_wikitext_text(split=ecfg['perplexity']['split'], subset=ecfg['perplexity']['subset'])
ppl_s = compute_perplexity(student, tok, text,
                           block_size=ecfg['perplexity']['block_size'],
                           stride=ecfg['perplexity']['stride'])
print(f'student PPL = {ppl_s:.3f}')

In [ ]:
lat_s = measure_latency(student, tok, **ecfg['latency'])
vram_s = peak_vram_gb()
print(f"student tok/s = {lat_s['tokens_per_sec']:.1f}  ttft = {lat_s['ttft_ms']:.1f} ms  peak VRAM = {vram_s:.2f} GB")

ctx = RunContext(
    tag='baseline-student', model=mcfg['student']['name'], quantization='none', distilled=False,
    perplexity=ppl_s, tokens_per_sec=lat_s['tokens_per_sec'], ttft_ms=lat_s['ttft_ms'],
    peak_vram_gb=vram_s,
)
save_metrics(new_run_dir(ctx.tag), ctx.to_dict())

In [ ]:
# Free VRAM before loading the 8B teacher.
del student
gc.collect()
torch.cuda.empty_cache()
print(f'after free: {torch.cuda.memory_allocated()/1024**3:.2f} GB allocated')

## 2 · Teacher baseline (Llama-3.1-8B)

On L4 (23 GB) fp16 (~16 GB) should fit with room for activations. If you hit OOM, change `quant='none'` → `quant='int8'` below — **but then record it as a separate run (`baseline-teacher-int8`)** so the Pareto plot shows both points.

In [ ]:
reset_peak()
TEACHER_QUANT = 'none'   # flip to 'int8' if fp16 OOMs on L4
teacher = load_model(mcfg['teacher']['name'], quant=TEACHER_QUANT, dtype=mcfg['teacher']['dtype'])
print('loaded teacher')

In [ ]:
ppl_t = compute_perplexity(teacher, tok, text,
                           block_size=ecfg['perplexity']['block_size'],
                           stride=ecfg['perplexity']['stride'])
print(f'teacher PPL = {ppl_t:.3f}')

In [ ]:
lat_t = measure_latency(teacher, tok, **ecfg['latency'])
vram_t = peak_vram_gb()
print(f"teacher tok/s = {lat_t['tokens_per_sec']:.1f}  ttft = {lat_t['ttft_ms']:.1f} ms  peak VRAM = {vram_t:.2f} GB")

tag = 'baseline-teacher' if TEACHER_QUANT == 'none' else f'baseline-teacher-{TEACHER_QUANT}'
ctx = RunContext(
    tag=tag, model=mcfg['teacher']['name'], quantization=TEACHER_QUANT, distilled=False,
    perplexity=ppl_t, tokens_per_sec=lat_t['tokens_per_sec'], ttft_ms=lat_t['ttft_ms'],
    peak_vram_gb=vram_t,
)
save_metrics(new_run_dir(ctx.tag), ctx.to_dict())

## 3 · Commit results back to GitHub

`results/runs/` is small JSON, tracked in git. Push so you can pull it locally and plot.

In [ ]:
!git -C $REPO config user.email 'colab@local'
!git -C $REPO config user.name 'colab'
!git -C $REPO add results/
!git -C $REPO commit -m 'Phase 1 baseline results'
!git -C $REPO push

### Expected ballpark (for sanity check)

| Model | PPL (WikiText-2) | tok/s (L4) | VRAM |
|---|---|---|---|
| Llama-3.2-1B fp16 | ~9–10 | ~60–90 | ~2.5 GB |
| Llama-3.1-8B fp16 | ~6–7 | ~15–25 | ~16 GB |

Numbers vary with `block_size`/`stride`. If PPL is way off (e.g. 50+), check tokenizer and eval config.